In [1]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
# =========================================================
# LOAD DATA
# =========================================================

train_df = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')

test_df = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

test_ids = test_df['PassengerId']

In [3]:
# =========================================================
# COMBINE DATA
# =========================================================

train_len = len(train_df)

combined = pd.concat(
    [train_df, test_df],
    axis=0
).reset_index(drop=True)

In [4]:
# =========================================================
# HANDLE MISSING VALUES CORRECTLY
# =========================================================

for col in combined.columns:

    # CATEGORICAL COLUMNS
    if (
        combined[col].dtype == 'object'
        or str(combined[col].dtype) == 'category'
        or combined[col].dtype == 'bool'
    ):

        combined[col] = combined[col].fillna('Unknown')

    # NUMERICAL COLUMNS
    else:

        combined[col] = combined[col].fillna(
            combined[col].median()
        )

In [5]:
# =========================================================
# DROP COLUMNS
# =========================================================

combined.drop(
    columns=[
        'PassengerId',
        'Cabin',
        'Name'
    ],
    inplace=True
)

In [6]:
# =========================================================
# SPLIT TRAIN TEST
# =========================================================

train_processed = combined.iloc[:train_len]

test_processed = combined.iloc[train_len:]

X = train_processed.drop('Transported', axis=1)

y = train_processed['Transported'].astype(int)

test_final = test_processed.drop('Transported', axis=1)

In [7]:
# =========================================================
# CATEGORICAL FEATURES
# =========================================================

cat_features = X.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

print(cat_features)

['HomePlanet', 'CryoSleep', 'Destination', 'VIP']


In [8]:
# =========================================================
# CATBOOST MODEL
# =========================================================

model = CatBoostClassifier(

    iterations=5000,

    learning_rate=0.02,

    depth=8,

    loss_function='Logloss',

    eval_metric='Accuracy',

    random_seed=42,

    verbose=500
)

In [9]:
# =========================================================
# STRATIFIED KFOLD
# =========================================================

skf = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)

scores = []

for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]

    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    y_valid = y.iloc[valid_idx]

    model.fit(

        X_train,

        y_train,

        cat_features=cat_features,

        eval_set=(X_valid, y_valid),

        verbose=False
    )

    preds = model.predict(X_valid)

    score = accuracy_score(y_valid, preds)

    scores.append(score)

    print(score)

print("Mean Accuracy :", np.mean(scores))

0.8125359401955147
0.7918343875790684
0.80448533640023
0.806674338319908
0.7917146144994246
Mean Accuracy : 0.8014489233988291


In [10]:
# =========================================================
# TRAIN FINAL MODEL
# =========================================================

model.fit(

    X,

    y,

    cat_features=cat_features,

    verbose=500
)

0:	learn: 0.7779823	total: 16.6ms	remaining: 1m 23s
500:	learn: 0.8195100	total: 7.27s	remaining: 1m 5s
1000:	learn: 0.8490740	total: 15.2s	remaining: 1m
1500:	learn: 0.8692051	total: 23.2s	remaining: 54.2s
2000:	learn: 0.8856551	total: 31.4s	remaining: 47s
2500:	learn: 0.8993443	total: 39.7s	remaining: 39.6s
3000:	learn: 0.9091223	total: 47.9s	remaining: 31.9s
3500:	learn: 0.9178649	total: 56.2s	remaining: 24.1s
4000:	learn: 0.9255723	total: 1m 4s	remaining: 16.1s
4500:	learn: 0.9293685	total: 1m 12s	remaining: 8.07s
4999:	learn: 0.9333947	total: 1m 21s	remaining: 0us


CatBoostClassifier(depth=8, eval_metric='Accuracy', iterations=5000, learning_rate=0.02, loss_function='Logloss', random_seed=42, verbose=500)

In [11]:
# =========================================================
# TEST PREDICTIONS
# =========================================================

test_predictions = model.predict(test_final)

In [12]:
# =========================================================
# CREATE SUBMISSION
# =========================================================

submission = pd.DataFrame({

    'PassengerId': test_ids,

    'Transported': test_predictions.astype(bool)

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv saved successfully 🚀")

submission.csv saved successfully 🚀
